In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------------
# STEP 1: Load dataset (only ToTensor)
# -----------------------------------
basic_transform = transforms.ToTensor()

train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=basic_transform
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=100,
    shuffle=False
)

# -----------------------------------
# STEP 2: Compute Mean and Std
# -----------------------------------
mean = torch.zeros(3)
std = torch.zeros(3)
total_images = 0

for images, _ in train_loader:
    batch_size = images.size(0)

    # Reshape: (B, C, H, W) -> (B, C, H*W)
    images = images.view(batch_size, 3, -1)

    mean += images.mean(dim=2).sum(dim=0)
    std += images.std(dim=2).sum(dim=0)

    total_images += batch_size

mean /= total_images
std /= total_images

print("Computed Mean:", mean)
print("Computed Std:", std)

# -----------------------------------
# STEP 3: Define Final Transform
# -----------------------------------
final_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# -----------------------------------
# STEP 4: Reload Dataset with Normalization
# -----------------------------------
train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=False,
    transform=final_transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=False,
    transform=final_transform
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

# -----------------------------------
# DONE: Ready for training
# -----------------------------------
print("Data is ready for training with normalization applied!")

# -----------------------------
# 2. Define Simple CNN
# -----------------------------
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()

        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)  # reduces size by half

        self.fc = nn.Linear(32 * 8 * 8, 10)  # after 2 poolings

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))  # 32 -> 16
        x = self.pool(self.relu(self.conv2(x)))  # 16 -> 8
        x = x.view(x.size(0), -1)               # flatten or FC layer
        x = self.fc(x)
        return x

model = SimpleCNN().to(device)

# -----------------------------
# 3. Loss and Optimizer
# -----------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# -----------------------------
# 4. Training (2–3 epochs demo)
# -----------------------------
for epoch in range(2):
    running_loss = 0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss:.4f}")

# -----------------------------
# 5. Testing
# -----------------------------
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")

100%|██████████| 170M/170M [01:21<00:00, 2.09MB/s]


Epoch 1, Loss: 1138.5644
Epoch 2, Loss: 884.5451
Test Accuracy: 63.00%
